# CH12 &mdash; A camera, a Sobel filter and a screenThe filter from CH11, moved into the video path.CH11's accelerator read a frame out of DDR, filtered it and wrote it back, and anotebook drove it one frame at a time: write the arguments, set `ap_start`, poll`ap_done`. This one never touches DDR. It sits inside the MIPI pipeline betweenthe colour-space converter and the packer, filtering pixels as they arrive:```Pcam 5C --MIPI CSI-2--> csi2_rx -> demosaic -> gamma -> CSC        -> [ sobel_stream ] -> pixel_pack -> VDMA -> DDR -> DPDMA -> DisplayPort```There is no `ap_start` and nothing to poll. The block free-runs off the stream,and the only thing this notebook does to it is write three registers.**Before you start**The filter can look at two things, and the notebook switches between them witha single register write:* the **camera**, straight off the MIPI receiver, and* **frames from DDR** &mdash; a video file, a still, or a test pattern &mdash;  played back into the video stream by the VDMA. The filter cannot tell the  difference: it is the identical 48-bit two-pixel stream either way.**Before you start*** A monitor on the DisplayPort. A Pcam 5C too, if you want the camera source;  the file source works without one.* `sobel_stream.bit`, `sobel_stream.hwh` and `sobel_stream.dtbo` in this  directory, all three with the same basename &mdash; PYNQ loads the device-tree  overlay by matching it against the bitstream name, and without it Linux never  creates the I2C adapter the camera is configured through.* `sobel_ref.py` and `video_source.py` from `CH12/sw/` next to this notebook.* The notebook has to run as root to program the PL and open the camera.

## Load the overlayThree names matter here. `ol.mipi` is the camera hierarchy, and PYNQ binds its`Pcam5C` driver to it by looking for `mipi_csi2_rx_subsyst`, `demosaic`,`gamma_lut`, `v_proc_sys`, `pixel_pack`, `gpio_ip_reset` and `axi_vdma` byname &mdash; which is why CH12's block design spells them exactly as AMD's baseoverlay does. `ol.mipi.sobel` is the filter. Constructing the hierarchy is whatbrings the camera up: `libpcam5c.so` runs the OV5640's I2C initialisation andconfigures the demosaic, the gamma LUT and the CSC.

In [ ]:
import sys, timeimport numpy as npimport PIL.Imagefrom pynq import Overlayfrom pynq.lib.video import VideoMode, DisplayPort, PIXEL_RGBsys.path.insert(0, '.')import sobel_refimport video_source as vsol    = Overlay('sobel_stream.bit')mipi  = ol.mipisobel = ol.mipi.sobelprint(sobel.register_map)

## Tell the filter the geometry, and start the cameraThree registers, and only three:| offset | register | ||---|---|---|| 0x10 | `img_width` | pixels per line. Must be even &mdash; two pixels arrive per clock, so an odd width has no representation on the bus || 0x18 | `img_height` | lines per frame || 0x20 | `mode` | 0 grayscale, 1 Sobel, 2 inverted, 3 colour passthrough |The width and height are not decoration. The filter counts its own way throughthe frame and needs to know where a line ends and where the last line is; givena geometry it cannot honour it drains the stream rather than filtering it, whichis the one failure mode that does not need a reset of the whole video pipelineto recover from.`mode` is latched at the start of each frame, so writing it from here can nevertear a frame in half.

In [ ]:
W, H = 1280, 720sobel.register_map.img_width  = Wsobel.register_map.img_height = Hsobel.register_map.mode       = sobel_ref.MODE_COLOR   # start with the raw cameravs.select_source(mipi, vs.SOURCE_CAMERA)   # the reset default, but say it anywaymipi.configure(VideoMode(W, H, 24))mipi.start()frame = mipi.readframe()print(frame.shape, frame.dtype)PIL.Image.fromarray(frame[:, :, ::-1])          # the stream is B,G,R; PIL wants R,G,B

## The four modesSwitching mode is a single register write, and it takes effect on the nextframe. Nothing else changes: same clock, same frame rate, same everything. Thefilter is in the pixel path, so it costs no time at all &mdash; a point thesecond notebook makes with a stopwatch.

In [ ]:
def grab(mode, settle=3):    """Set a mode and return a frame taken after it has taken effect.    The mode is latched at start-of-frame, and the VDMA is several frames deep,    so a couple of frames have to go by before what comes back is what was    asked for."""    sobel.register_map.mode = mode    for _ in range(settle):        f = mipi.readframe()    return fshots = {name: grab(mode) for mode, name in sobel_ref.MODE_NAMES.items()}strip = np.hstack([shots[n][::2, ::2] for n in                   ('colour', 'grayscale', 'sobel', 'inverted')])PIL.Image.fromarray(strip[:, :, ::-1])

## A word about 1080p30The filter handles either mode the Pcam 5C offers &mdash; its line buffers aresized for 1920 pixels &mdash; but PYNQ's `Pcam5C` driver does not offer you thechoice. It calls `libpcam5c.so` once, when the hierarchy is constructed, withthe mode hardcoded:```pythonself._handle = pcam5c_lib.pcam_mipi(i2c_index,                                    int(MIPIMode.r1280x720_60.value), ...)````mipi.configure(VideoMode(1920, 1080, 24))` changes what the VDMA expects butnot what the sensor sends, so it is not enough on its own. To actually get1080p30, call the same entry point again with the other mode and tell the filterabout it. This reaches around the driver into the library it wraps, so treat itas what it is &mdash; a workaround for a hardcoded constant, not an API.

In [ ]:
def set_camera_mode(usermode, width, height):    """Re-run the C bring-up with a different sensor mode.    usermode 0 = 1280x720@60, 1 = 1920x1080@30 (pynq.lib.video.pcam5c.MIPIMode).    """    import cffi, glob, os    from pynq.lib.video.constants import LIB_SEARCH_PATH    ffi = cffi.FFI()    ffi.cdef('int pcam_mipi(int, int, unsigned long, unsigned long, '             'unsigned long, unsigned long);')    lib = ffi.dlopen(os.path.join(LIB_SEARCH_PATH, 'libpcam5c.so'))    i2c = 6    for dev in glob.glob('/dev/i2c-*'):        n = os.path.basename(dev).split('-')[-1]        try:            with open(f'/sys/bus/i2c/devices/i2c-{n}/of_node/label') as fh:                if 'RPICAM' in fh.read():                    i2c = int(n)        except FileNotFoundError:            continue    mipi.stop()    rc = lib.pcam_mipi(i2c, usermode,                       mipi.gpio_ip_reset.mmio.array.ctypes.data,                       mipi.v_proc_sys.mmio.array.ctypes.data,                       mipi.gamma_lut.mmio.array.ctypes.data,                       mipi.demosaic.mmio.array.ctypes.data)    if rc < 0:        raise RuntimeError('camera re-initialisation failed')    sobel.register_map.img_width  = width    sobel.register_map.img_height = height    mipi.configure(VideoMode(width, height, 24))    mipi.start()    return mipi.readframe()# set_camera_mode(1, 1920, 1080)     # uncomment for 1080p30

## Out to the screenThe camera stream is B,G,R and the DisplayPort is configured for R,G,B, so thechannels are reversed on the way out. That reversal is the only per-pixel workthe A53s do in this loop; everything else is a copy.

In [ ]:
dp = DisplayPort()dp.configure(VideoMode(W, H, 24), PIXEL_RGB)sobel.register_map.mode = sobel_ref.MODE_SOBELframes = 120t0 = time.perf_counter()for _ in range(frames):    f  = mipi.readframe()    out = dp.newframe()    out[:] = f[:, :, ::-1]    dp.writeframe(out)elapsed = time.perf_counter() - t0print(f'{frames} frames in {elapsed:.2f}s = {frames/elapsed:.1f} fps')

## Cycle the modes on the screenWatch the monitor rather than the notebook for this one.

In [ ]:
for mode, name in sobel_ref.MODE_NAMES.items():    sobel.register_map.mode = mode    print(f'{name} ...')    t_end = time.time() + 3    while time.time() < t_end:        f = mipi.readframe()        out = dp.newframe()        out[:] = f[:, :, ::-1]        dp.writeframe(out)

## The other source: frames from DDRThe filter has no memory port. It is a streaming block wired between thecolour-space converter and the packer, so "run this video file through it" isnot something software can arrange on its own &mdash; there has to be a path inthe PL that plays frames from DDR into the video stream. There is one:```camera --> csi2_rx -> demosaic -> gamma -> CSC -> channel_swap --\                                                                  axis_switch --> sobel --> pixel_pack --> VDMA S2MM --> DDRDDR --> VDMA MM2S -> pixel_unpack ------------------------------/````pixel_unpack` is the mirror of the packer already in the pipeline: 64-bit wordsout of DDR back into 48-bit two-pixel beats, TUSER and TLAST intact. The switchpicks which one the filter sees.A two-into-one AXI4-Stream switch has no AXI4-Lite register map &mdash; thatonly appears when there is more than one master to route to &mdash; so thechoice is made through `s_req_suppress`, one bit per input, driven from a smallGPIO. Suppressing an input stops the arbiter ever granting it. `video_source.py`wraps that; the reset default is the camera.One thing to know: the suppressed source is backpressured, not switched off. Itspixels have nowhere to go, so with the camera deselected the CSI-2 receiver'sline buffer fills and it starts flagging overflow. Nothing breaks &mdash; itresynchronises at the next frame &mdash; but for a long stretch of playback itis tidier to turn the receiver off.

In [ ]:
player = vs.FramePlayer(mipi, W, H)player.start()                       # selects the file source as a side effectvs.camera_enabled(mipi, False)       # stop the camera shouting into a closed doorpattern = vs.test_pattern(W, H)sobel.register_map.mode = sobel_ref.MODE_SOBELfor _ in range(4):                   # let the VDMA get a few frames deep    player.play(pattern)    out = mipi.readframe()shown = dp.newframe()shown[:] = out[:, :, ::-1]dp.writeframe(shown)                 # the monitor is showing it tooPIL.Image.fromarray(np.hstack([pattern[::2, ::2], out[::2, ::2]])[:, :, ::-1])

## Now the filter can be checked exactlyWith the camera there is no way to know what went into the filter. The sensor isstill exposing, auto-exposure and auto-white-balance are still moving, and theframe before the one you filtered is not the frame you filtered &mdash; so thebest that can be done is an approximate comparison, which is what the lastsection of this notebook does.Playing a known pattern in removes all of that. The input is a constant, thesoftware reference can be applied to exactly it, and the answer should be**zero** differing samples in every mode. This is the same check the RTLtestbench makes in simulation, made again on the real thing.

In [ ]:
for mode, name in sobel_ref.MODE_NAMES.items():    sobel.register_map.mode = mode    for _ in range(4):               # flush the mode change through the VDMA        player.play(pattern)        got = mipi.readframe()    expected = sobel_ref.filter_frame(pattern, mode)    bad, worst = sobel_ref.compare(expected, got)    verdict = 'EXACT' if bad == 0 else f'{bad} samples differ, worst {worst}'    print(f'{name:<10} {verdict}')

Anything other than `EXACT` here is a real disagreement between the hardware andthe model, not sensor noise &mdash; worth chasing.

## Playing a video fileThe same path, with frames from a file instead of a pattern. Frames have toarrive as `(H, W, 3)` uint8 in B,G,R order, which is what OpenCV's `VideoCapture`already gives you, and the width has to be a multiple of 8 &mdash; `pixel_unpack`turns three 64-bit words into four beats, so a line has to be a whole number ofthose groups. Both camera resolutions are.If the board's OpenCV was built without video codecs, use the still-image cellbelow instead; it exercises exactly the same hardware path.

In [ ]:
import cv2CLIP = 'clip.mp4'                    # put a file next to this notebookcap = cv2.VideoCapture(CLIP)if not cap.isOpened():    raise RuntimeError(f'could not open {CLIP}')sobel.register_map.mode = sobel_ref.MODE_SOBELframes = 0t0 = time.perf_counter()while True:    ok, frame = cap.read()    if not ok:        break    if frame.shape[:2] != (H, W):        frame = cv2.resize(frame, (W, H))    player.play(np.ascontiguousarray(frame))    out = mipi.readframe()           # the filtered frame, straight back from DDR    shown = dp.newframe()            # ...and on to the monitor    shown[:] = out[:, :, ::-1]    dp.writeframe(shown)    frames += 1cap.release()print(f'{frames} frames through the filter in {time.perf_counter()-t0:.2f}s')PIL.Image.fromarray(out[:, :, ::-1])

### ...or a still imageNo codecs needed, and it is the easier thing to look at when checking that thefilter is doing what you expect.

In [ ]:
img = PIL.Image.open('test.jpg').convert('RGB').resize((W, H))still = np.asarray(img)[:, :, ::-1].copy()      # PIL is R,G,B; the stream is B,G,Rshots = {}for mode, name in sobel_ref.MODE_NAMES.items():    sobel.register_map.mode = mode    for _ in range(4):        player.play(still)        shots[name] = mipi.readframe().copy()strip = np.hstack([shots[n][::2, ::2] for n in                   ('colour', 'grayscale', 'sobel', 'inverted')])PIL.Image.fromarray(strip[:, :, ::-1])

## Back to the cameraStopping the player restores the camera source; the receiver has to be switchedback on separately.

In [ ]:
player.stop()                        # also selects SOURCE_CAMERAvs.camera_enabled(mipi, True)print('source is', 'camera' if vs.current_source(mipi) == vs.SOURCE_CAMERA else 'file')sobel.register_map.mode = sobel_ref.MODE_SOBELframe = mipi.readframe()PIL.Image.fromarray(frame[:, :, ::-1])

## Does the hardware agree with the software?The definitive answer to that question is in simulation, where the sametestbench drives the HLS output, the SystemVerilog and the VHDL against the samegolden model and all three match it exactly. On hardware the comparison isharder than it looks: the filter is *inside* the pipeline, so there is no way tocapture the pixels going into it and the pixels coming out of it for the sameframe. The best that can be done is to grab a raw frame, switch mode, grab afiltered one, and compare the second against the software reference applied tothe first.That is only meaningful if the scene and the sensor hold still in between, whichwith auto-exposure and auto-white-balance running is not a given. So the checkstarts by proving they did: two consecutive raw frames have to be *identical*before the comparison is worth reporting. Point the camera at something staticand well lit, and give the exposure a few seconds to settle.

In [ ]:
# 1. is the sensor holding still?sobel.register_map.mode = sobel_ref.MODE_COLORfor _ in range(5):    mipi.readframe()a = mipi.readframe().copy()b = mipi.readframe().copy()drift, worst = sobel_ref.compare(a, b)print(f'two consecutive raw frames differ in {drift} of {a.size} samples '      f'(worst {worst})')if drift:    print('the scene or the sensor is still moving -- the comparison below '          'will show that, not a hardware error')

In [ ]:
# 2. compare each mode against the software reference applied to frame `a`for mode, name in sobel_ref.MODE_NAMES.items():    got      = grab(mode, settle=4)    expected = sobel_ref.filter_frame(a, mode)    bad, worst = sobel_ref.compare(expected, got)    pct = 100.0 * bad / got.size    print(f'{name:<10} {bad:8d} samples differ ({pct:5.2f}%), worst {worst:3d}')

A static, well-lit scene lands within a bit or two on a small fraction of thesamples &mdash; that residue is the sensor, not the filter. If it does not, lookat the drift number above before suspecting the hardware.

## Tidy upThe DisplayPort and the camera both hold buffers; leaving them held means thenext notebook cannot open them.

In [ ]:
dp.close()mipi.stop()print('released')